# Python de 0 a experto — 6. Type hints, dataclasses y decoradores

## Introducción
Contenido avanzado: tres herramientas que aparecen todo el tiempo en
código Python profesional moderno y que **preparan el terreno para POO,
SOLID y Patrones de Diseño** (próximas sesiones): anotaciones de tipo,
`@dataclass` y decoradores (propios y los que ya conoces como
`@staticmethod`/`@property`).

## Objetivos
- Anotar tipos en variables y funciones con el módulo `typing`.
- Usar `@dataclass` para eliminar código repetitivo (`__init__`, `__repr__`,
  `__eq__`) en clases que solo almacenan datos.
- Explicar qué es un decorador y escribir uno propio, con y sin argumentos.
- Diferenciar `@staticmethod`/`@property` (decoradores ya conocidos de
  clases) de un decorador de función genérico.

## 1. Type hints

Las anotaciones de tipo **no son obligatorias ni se verifican en tiempo de
ejecución** — Python las ignora al correr el programa. Sirven para que el
editor/IDE, herramientas como `mypy` y, sobre todo, quien lea tu código
después, sepan qué tipo se espera. Es documentación ejecutable.

In [1]:
from typing import Optional, Union

def dividir(a: float, b: float) -> Optional[float]:
    """Devuelve None si no se puede dividir, en vez de lanzar excepción."""
    if b == 0:
        return None
    return a / b

def procesar_id(identificador: Union[int, str]) -> str:
    """Acepta un id como int o como str, siempre devuelve str."""
    return f"ID-{identificador}"

print(dividir(10, 2), dividir(10, 0))
print(procesar_id(42), procesar_id("42B"))

# Python 3.10+ permite '|' en vez de Union/Optional
def saludar(nombre: str | None = None) -> str:
    return f"Hola, {nombre or 'desconocido'}"

print(saludar("Ana"), "|", saludar())

5.0 None
ID-42 ID-42B
Hola, Ana | Hola, desconocido


In [2]:
from typing import List, Dict

def promedio(notas: List[float]) -> float:
    return sum(notas) / len(notas) if notas else 0.0

def contar_palabras(texto: str) -> Dict[str, int]:
    conteo: Dict[str, int] = {}
    for palabra in texto.lower().split():
        conteo[palabra] = conteo.get(palabra, 0) + 1
    return conteo

print(promedio([4.5, 3.8, 5.0]))
print(contar_palabras("python es genial python es claro"))

# variable anotada explícitamente, sin asignar en la misma línea
edad_minima: int
edad_minima = 18
print(edad_minima, type(edad_minima))

4.433333333333334
{'python': 2, 'es': 2, 'genial': 1, 'claro': 1}
18 <class 'int'>


## 2. `@dataclass` — clases que solo guardan datos, sin boilerplate

Comparación: una clase "normal" para representar un dato requiere escribir
a mano `__init__`, `__repr__` y `__eq__` si quieres que se comparen por
valor. `@dataclass` los genera automáticamente a partir de las
anotaciones de los atributos.

In [3]:
# --- SIN dataclass: clase normal ---
class ProductoNormal:
    def __init__(self, nombre: str, precio: float, stock: int = 0):
        self.nombre = nombre
        self.precio = precio
        self.stock = stock

    def __repr__(self):
        return f"ProductoNormal(nombre={self.nombre!r}, precio={self.precio!r}, stock={self.stock!r})"

    def __eq__(self, otro):
        if not isinstance(otro, ProductoNormal):
            return NotImplemented
        return (self.nombre, self.precio, self.stock) == (otro.nombre, otro.precio, otro.stock)


p1 = ProductoNormal("Teclado", 120000, 10)
p2 = ProductoNormal("Teclado", 120000, 10)
print(p1)
print("¿p1 == p2?", p1 == p2)

ProductoNormal(nombre='Teclado', precio=120000, stock=10)
¿p1 == p2? True


In [4]:
# --- CON dataclass: misma idea, sin boilerplate ---
from dataclasses import dataclass

@dataclass
class Producto:
    nombre: str
    precio: float
    stock: int = 0

p3 = Producto("Teclado", 120000, 10)
p4 = Producto("Teclado", 120000, 10)
print(p3)                     # __repr__ generado automáticamente
print("¿p3 == p4?", p3 == p4)  # __eq__ generado automáticamente (compara por valor)

# también se puede pedir inmutabilidad, orden, etc.
@dataclass(frozen=True)
class Punto:
    x: float
    y: float

pt = Punto(1.0, 2.0)
print(pt)
try:
    pt.x = 99  # frozen=True: no se puede reasignar, como una tupla
except Exception as e:
    print("Error esperado:", type(e).__name__, e)

Producto(nombre='Teclado', precio=120000, stock=10)
¿p3 == p4? True
Punto(x=1.0, y=2.0)
Error esperado: FrozenInstanceError cannot assign to field 'x'


`@dataclass` reduce drásticamente el código repetitivo, pero **no
valida nada**: `Producto("Teclado", "gratis", -5)` es perfectamente válido
para Python en tiempo de ejecución aunque el precio sea un string y el
stock sea negativo. Esa limitación es justo lo que resuelve Pydantic, en
el siguiente notebook.

## 3. Decoradores

Un decorador es una función que **recibe una función y devuelve otra
función** (normalmente envolviéndola con comportamiento extra), usando la
sintaxis `@nombre_decorador` encima de la definición.

In [5]:
import time

def medir_tiempo(func):
    def wrapper(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = func(*args, **kwargs)
        duracion = time.perf_counter() - inicio
        print(f"[{func.__name__}] tardó {duracion:.6f} s")
        return resultado
    return wrapper


@medir_tiempo
def suma_lenta(n):
    return sum(range(n))


print(suma_lenta(1_000_000))
# @medir_tiempo es azúcar sintáctico equivalente a:
#   suma_lenta = medir_tiempo(suma_lenta)

[suma_lenta] tardó 0.014355 s
499999500000


### Decoradores con argumentos propios

Para que el decorador reciba argumentos (por ejemplo, cuántas veces
repetir algo), se necesita una capa extra de función: una "fábrica de
decoradores".

In [6]:
def repetir(veces: int):
    def decorador(func):
        def wrapper(*args, **kwargs):
            resultados = []
            for _ in range(veces):
                resultados.append(func(*args, **kwargs))
            return resultados
        return wrapper
    return decorador


@repetir(veces=3)
def lanzar_dado():
    import random
    return random.randint(1, 6)


random_seed_demo = lanzar_dado()
print("3 lanzamientos:", random_seed_demo)

3 lanzamientos: [3, 5, 1]


### Decoradores que ya conoces: `@staticmethod` y `@property`

Son decoradores integrados que se usan **dentro de una clase**:
- `@staticmethod`: el método no recibe `self`, es una función agrupada por
  contexto dentro de la clase (no depende de la instancia).
- `@property`: convierte un método en un atributo de solo lectura
  (se accede sin paréntesis), útil para exponer un valor calculado.

Se mencionan aquí porque son la prueba de que un decorador puede venir del
propio lenguaje (`staticmethod`, `property`) o ser escrito por ti
(`medir_tiempo`, `repetir` arriba) — es exactamente el mismo mecanismo.

In [7]:
class Circulo:
    def __init__(self, radio: float):
        self.radio = radio

    @property
    def area(self) -> float:
        return 3.1416 * self.radio ** 2

    @staticmethod
    def es_radio_valido(radio: float) -> bool:
        return radio > 0


c = Circulo(3)
print("área (como atributo, sin paréntesis):", c.area)
print("¿-2 es un radio válido?", Circulo.es_radio_valido(-2))

área (como atributo, sin paréntesis): 28.2744
¿-2 es un radio válido? False


## Ejercicios prácticos

1. Anota con type hints una función `filtrar_mayores(edades: list[int],
   limite: int) -> list[int]` que devuelva las edades mayores al límite.
2. Convierte esta clase normal a `@dataclass`:
   ```python
   class Libro:
       def __init__(self, titulo, autor, anio):
           self.titulo = titulo
           self.autor = autor
           self.anio = anio
   ```
3. Escribe un decorador `@solo_positivos` que valide que todos los
   argumentos posicionales de la función decorada sean números positivos,
   lanzando `ValueError` si no.
4. Escribe una fábrica de decoradores `@reintentar(intentos=3)` que vuelva
   a ejecutar la función si lanza una excepción, hasta agotar los
   intentos.

## Autoevaluación

- ¿Los type hints impiden pasar un tipo incorrecto en tiempo de ejecución?
  ¿Por qué sí o por qué no?
- ¿Qué métodos genera automáticamente `@dataclass` que tendrías que
  escribir a mano en una clase normal?
- ¿Por qué un decorador que acepta argumentos propios necesita una función
  extra (una "fábrica")?

## Referencias
- [Documentación oficial — módulo `typing`](https://docs.python.org/es/3/library/typing.html)
- [Documentación oficial — módulo `dataclasses`](https://docs.python.org/es/3/library/dataclasses.html)
- [Real Python — Primer on Python Decorators](https://realpython.com/primer-on-python-decorators/)